# MPIA Arxiv on Deck 2

Contains the steps to produce the paper extractions.

In [1]:
# Imports
import os
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import warnings
from PIL import Image 
import re

# requires arxiv_on_deck_2

from arxiv_on_deck_2.arxiv2 import (get_new_papers, 
                                    get_paper_from_identifier,
                                    retrieve_document_source, 
                                    get_markdown_badge)
from arxiv_on_deck_2 import (latex,
                             latex_bib,
                             mpia,
                             highlight_authors_in_list)

# Sometimes images are really big
Image.MAX_IMAGE_PIXELS = 1000000000 

In [2]:
# Some useful definitions.

class AffiliationWarning(UserWarning):
    pass

class AffiliationError(RuntimeError):
    pass

def validation(source: str):
    """Raises error paper during parsing of source file
    
    Allows checks before parsing TeX code.
    
    Raises AffiliationWarning
    """
    check = mpia.affiliation_verifications(source, verbose=True)
    if check is not True:
        raise AffiliationError("mpia.affiliation_verifications: " + check)

        
warnings.simplefilter('always', AffiliationWarning)


def get_markdown_qrcode(paper_id: str):
    """ Generate a qrcode to the arxiv page using qrserver.com
    
    :param paper: Arxiv paper
    :returns: markdown text
    """
    url = r"https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="
    txt = f"""<img src={url}"https://arxiv.org/abs/{paper_id}">"""
    txt = '<div id="qrcode">' + txt + '</div>'
    return txt


def clean_non_western_encoded_characters_commands(text: str) -> str:
    """ Remove non-western encoded characters from a string
    List may need to grow.
    
    :param text: the text to clean
    :return: the cleaned text
    """
    text = re.sub(r"(\\begin{CJK}{UTF8}{gbsn})(.*?)(\\end{CJK})", r"\2", text)
    return text


def get_initials(name: str) -> str:
    """ Get the short name, e.g., A.-B. FamName
    :param name: full name
    :returns: initials
    """
    initials = []
    # account for non western names often in ()
    if '(' in name:
        name = clean_non_western_encoded_characters_commands(name)
        suffix = re.findall(r"\((.*?)\)", name)[0]
        name = name.replace(f"({suffix})", '')
    else:
        suffix = ''
    split = name.split()
    for token in split[:-1]:
        if '-' in token:
            current = '-'.join([k[0] + '.' for k in token.split('-')])
        else:
            current = token[0] + '.'
        initials.append(current)
    initials.append(split[-1].strip())
    if suffix:
        initials.append(f"({suffix})")
    return ' '.join(initials)

## get list of arxiv paper candidates

We use the MPIA mitarbeiter list webpage from mpia.de to get author names
We then get all new papers from Arxiv and match authors

In [3]:
# deal with the author list and edge cases of people that cannot be consistent on their name  

def filter_non_scientists(name: str) -> bool:
    """ Loose filter on expected authorships

    removing IT, administration, technical staff
    :param name: name
    :returns: False if name is not a scientist
    """
    remove_list = ['Licht', 'Binroth', 'Witzel', 'Jordan',
                   'Zähringer', 'Scheerer', 'Hoffmann', 'Düe',
                   'Hellmich', 'Enkler-Scharpegge', 'Witte-Nguy',
                   'Dehen', 'Beckmann', 'Jager', 'Jäger'
                  ]

    for k in remove_list:
        if k in name:
            return False
    return True

def add_author_to_list(author_list: list) -> list:
    """ Add author to list if not already in list
    
    :param author: author name
    :param author_list: list of authors
    :returns: updated list of authors
    """
    add_list = ['T. Henning', 'R. E. Hviding']

    for author in add_list:
        if author not in author_list:
            author_list.append(author)
    return author_list

# get list from MPIA website
# filter for non-scientists (mpia.get_mpia_mitarbeiter_list() does some filtering)
mpia_authors = [k[1] for k in mpia.get_mpia_mitarbeiter_list() if filter_non_scientists(k[1])]
# add some missing author because of inconsistencies in their MPIA name and author name on papers
mpia_authors = add_author_to_list(mpia_authors)

In [4]:
new_papers = get_new_papers()
# add manual references
add_paper_refs = []
new_papers.extend([get_paper_from_identifier(k) for k in add_paper_refs])

def robust_call(fn, value, *args, **kwargs):
    try:
        return fn(value, *args, **kwargs)
    except Exception:
        return value

candidates = []
for paperk in new_papers:
    # Check author list with their initials
    normed_author_list = [robust_call(mpia.get_initials, k) for k in paperk['authors']]
    hl_authors = highlight_authors_in_list(normed_author_list, mpia_authors, verbose=True)
    matches = [(hl, orig) for hl, orig in zip(hl_authors, paperk['authors']) if 'mark' in hl]
    paperk['authors'] = hl_authors
    if matches:
        # only select paper if an author matched our list
        candidates.append(paperk)
print("""Arxiv has {0:,d} new papers today""".format(len(new_papers)))        
print("""          {0:,d} with possible author matches""".format(len(candidates)))

M. Zhang  ->  M. Zhang  |  ['M. Zhang']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
Y. Zhang  ->  Y. Zhang  |  ['Y. Zhang']
J. Li  ->  J. Li  |  ['J. Li']
X. Lin  ->  X. Lin  |  ['X. Lin']
J. Li  ->  J. Li  |  ['J. Li']
X. Lin  ->  X. Lin  |  ['X. Lin']
M. M.lin  ->  M. Malin  |  ['M. Malin']
Y. Wang  ->  Y. Wang  |  ['Y. Wang']
S. Kraus  ->  S. Kraus  |  ['S. Kraus']
P. Smith  ->  P. Smith  |  ['P. Smith']
J. Li  ->  J. Li  |  ['J. Li']
H. Zhang  ->  H. Zhang  |  ['H. Zhang']
L. Mohr  ->  L. Mohr  |  ['L. Mohr']
M. V. Pallauta  ->  M. V. Pallauta  |  ['M. V. Pallauta']
Arxiv has 71 new papers today
          12 with possible author matches


# Parse sources and generate relevant outputs

From the candidates, we do the following steps:
* get their tarball from ArXiv (and extract data)
* find the main .tex file: find one with \documentclass{...} (sometimes it's non trivial)
* Check affiliations with :func:`validation`, which uses :func:`mpia.affiliation_verifications`
* If passing the affiliations: we parse the .tex source
   * inject sub-documents into the main (flatten the main document)
   * parse structure, extract information (title, abstract, authors, figures...)
   * handles `\graphicspath` if provided
* Generate the .md document.

In [5]:
documents = []
failed = []
for paper in tqdm(candidates):
    # debug crap
    paper['identifier'] = paper['identifier'].lower().replace('arxiv:', '').replace(r'\n', '').strip()
    paper_id = paper['identifier']
    
    folder = f'tmp_{paper_id}'

    try:
        if not os.path.isdir(folder):
            folder = retrieve_document_source(f"{paper_id}", f'tmp_{paper_id}')
        
        try:
            doc = latex.LatexDocument(folder, validation=validation)    
        except AffiliationError as affilerror:
            msg = f"ArXiv:{paper_id:s} is not an MPIA paper... " + str(affilerror)
            failed.append((paper, "affiliation error: " + str(affilerror) ))
            continue
        
        # Hack because sometimes author parsing does not work well
        if (len(doc.authors) != len(paper['authors'])):
            doc._authors = paper['authors']
        else:
            # highlight authors (FIXME: doc.highlight_authors)
            # done on arxiv paper already
            doc._authors = highlight_authors_in_list(
                [get_initials(k) for k in doc.authors], 
                mpia_authors, verbose=True)
        if (doc.abstract) in (None, ''):
            doc._abstract = paper['abstract']
            
        doc.comment = (get_markdown_badge(paper_id) + 
                       "<mark>Appeared on: " + paper['date'] + "</mark> - ")
        if paper['comments']:
            doc.comment += " _" + paper['comments'] + "_"
        
        full_md = doc.generate_markdown_text()
        
        full_md += get_markdown_qrcode(paper_id)
        
        # replace citations
        try:
            bibdata = latex_bib.LatexBib.from_doc(doc)
            full_md = latex_bib.replace_citations(full_md, bibdata)
        except Exception as e:
            print("Issues with the citations")
            print(e)
        
        documents.append((paper_id, full_md))
    except Exception as e:
        warnings.warn(latex.LatexWarning(f"{paper_id:s} did not run properly\n" +
                                         str(e)
                                        ))
        failed.append((paper, "latex error " + str(e)))

  0%|          | 0/12 [00:00<?, ?it/s]

Retrieving document from  https://arxiv.org/e-print/2611.26680


extracting tarball to tmp_2611.26680... done.
Retrieving document from  https://arxiv.org/e-print/2611.26682
extracting tarball to tmp_2611.26682...

 done.
Retrieving document from  https://arxiv.org/e-print/2611.26684


extracting tarball to tmp_2611.26684... done.
Retrieving document from  https://arxiv.org/e-print/2611.26685


extracting tarball to tmp_2611.26685... done.
  0: tmp_2611.26685/main.tex, 687 lines
  1: tmp_2611.26685/aassymbols.tex, 579 lines
Retrieving document from  https://arxiv.org/e-print/2611.26710


... (11 outputs left out)

 done.
Retrieving document from  https://arxiv.org/e-print/2611.27551


extracting tarball to tmp_2611.27551...

 done.
Retrieving document from  https://arxiv.org/e-print/2611.27643
extracting tarball to tmp_2611.27643...

 done.


### Export the logs

Throughout, we also keep track of the logs per paper. see `logs-{today date}.md` 

In [6]:
import datetime
today = str(datetime.date.today())
logfile = f"_build/html/logs/log-{today}.md"


with open(logfile, 'w') as logs:
    # Success
    logs.write(f'# Arxiv on Deck 2: Logs - {today}\n\n')
    logs.write("""* Arxiv had {0:,d} new papers\n""".format(len(new_papers)))
    logs.write("""    * {0:,d} with possible author matches\n\n""".format(len(candidates)))
    logs.write("## Sucessful papers\n\n")
    display(Markdown("## Successful papers"))
    success = [k[0] for k in documents]
    for candid in candidates:
        if candid['identifier'].split(':')[-1] in success:
            display(candid)
            logs.write(candid.generate_markdown_text() + '\n\n')

    ## failed
    logs.write("## Failed papers\n\n")
    display(Markdown("## Failed papers"))
    failed = sorted(failed, key=lambda x: x[1])
    current_reason = ""
    for paper, reason in failed:
        if 'affiliation' in reason:
            color = 'green'
        else:
            color = 'red'
        data = Markdown(
                paper.generate_markdown_text() + 
                f'\n|<p style="color:{color:s}"> **ERROR** </p>| <p style="color:{color:s}">{reason:s}</p> |'
               )
        if reason != current_reason:
            logs.write(f'### {reason:s} \n\n')
            current_reason = reason
        logs.write(data.data + '\n\n')
        
        # only display here the important errors (all in logs)
        # if color in ('red',):
        display(data)

## Successful papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.27082-b31b1b.svg)](https://arxiv.org/abs/2611.27082) | **IGRINS-2 Reveals an Oxygen-rich Dayside Atmosphere of WASP-12 b: Resolving the C/O Controversy**  |
|| C. Kye, et al. -- incl., <mark>P. Smith</mark> |
|*Appeared on*| *2026-11-23*|
|*Comments*| *12 pages, 8 figures, 1 table. Accepted for publication in ApJL*|
|**Abstract**|            Recent studies have shown that measurements of hot-Jupiter atmospheric composition, including carbon-to-oxygen (C/O) and refractory-to-volatile (R/V) abundance ratios, provide powerful diagnostics of the chemistry and formation of gas giant planets. Important questions remain about the atmospheric composition of the ultra-hot Jupiter WASP-12 b: its dayside C/O ratio remains under debate, and its R/V ratio is not yet well constrained. We present high-resolution (R $\sim$ 45,000) near-infrared dayside emission spectroscopy of the ultra-hot Jupiter WASP-12 b with IGRINS-2 on Gemini North during post-eclipse phases. Cross-correlation analysis reveals volatile molecular signals of H$_2$O and CO together with tentative signals of OH and gas-phase refractories in the planetary atmosphere. The planetary signals appear in emission, indicating a thermal inversion in the dayside atmosphere. A chemically consistent Bayesian retrieval yields C/O $=0.72^{+0.08}_{-0.10}$, disfavoring carbon-rich dayside solutions with C/O $>1$ at greater than $5\sigma$. This independent, line-resolved measurement supports an oxygen-rich atmosphere, providing a new constraint on the long-standing C/O controversy surrounding WASP-12 b. Our retrieval also yields a host-star-normalized R/V ratio, $[\mathrm{R/V}]_{*}=0.17^{+0.22}_{-0.23}$, which is consistent with those of other ultra-hot Jupiters similarly characterized with IGRINS/IGRINS-2. Although the sample remains small, this consistency suggests a shared atmospheric chemistry among these planets, potentially reflecting similar proportions of accreted volatile and refractory material.         |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.27643-b31b1b.svg)](https://arxiv.org/abs/2611.27643) | **Hints of planet formation in progress around DR Tau from observations with LBT SHARK-NIR and SHARK-VIS**  |
|| D. Mesa, et al. -- incl., <mark>L. Mohr</mark>, <mark>M. V. Pallauta</mark> |
|*Appeared on*| *2026-11-23*|
|*Comments*| *12 pages; 10 Figures; Accepted for publications on A&A*|
|**Abstract**|            Protoplanetary disks are the primary sites of planet formation. However, the direct imaging of protoplanets still embedded within their parental disks remains a significant challenge, with only a small number of confirmed detections to date and several candidates awaiting verification. We search for embedded planets in the protoplanetary disk of the young star DRTau by targeting accretion signatures from forming sub-stellar objects. Using narrow-band imaging, we observe both visible Halpha and near-infrared Paschen-beta (Pabeta) emission to identify actively accreting companions. To this end, we performed high-contrast observations of DRTau using the SHARK-VIS (0.40-0.95 micron) and SHARK-NIR (0.96-1.70 micron) imagers at the LBT. The study uses a combination of narrow-band filters to isolate accretion signatures and broad-band filters for multi-wavelength characterization in both the visible and near-infrared. Our observations recover the primary disk structures previously identified in the literature, albeit at low signal-to-noise (S/N). We identify two compact features with S/N between 6 and 13 in both narrow- and broad-band filters. The first, an elongated structure, matches previous SPHERE detections, while the second, more compact feature, is now confirmed despite its low S/N in earlier datasets. A comparison over a seven-year baseline confirms the stability of both structures. Their observed orbital motion is consistent with Keplerian rotation, potentially suggesting the presence of an undetected planetary-mass companion (few MJup) at the location of the second feature. However, our analysis does not confirm Halpha or Pabeta emission originating from localized accretion. Instead, we conclude that the observed flux is likely stellar light reflected by dust at these locations and discuss alternative mechanisms that could account for such detections.         |

## Failed papers


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.26680-b31b1b.svg)](https://arxiv.org/abs/2611.26680) | **PLATON 7: An ultra-fast exoplanet atmospheric retrieval code**  |
|| <mark>M. Zhang</mark>, et al. |
|*Appeared on*| *2026-11-23*|
|*Comments*| *Submitted to JOSS*|
|**Abstract**|            PLATON is an open source Python package for atmospheric retrieval, first released in 2018. We introduce PLATON 7, a complete rewrite of PLATON using Claude Fable 5. The biggest changes are that PLATON now uses JAX for all intensive calculations, and these calculations are now performed in single precision instead of double precision. These two changes speed up the code by a factor of ~60 with negligible decrease in accuracy, allowing it to compute a 0.8--5.3 um transmission spectrum at R=20k in 0.35 ms on a RTX 5090. We also introduce several new features, including support for two-limb retrievals, patchy clouds, and more opacities (particularly OCS and CS2). PLATON is available on GitHub at this https URL.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.26682-b31b1b.svg)](https://arxiv.org/abs/2611.26682) | **Blazars through the eyes of REDSoX and GOSoX**  |
|| S. Capecchiacci, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-23*|
|*Comments*| **|
|**Abstract**|            Blazar polarimetric properties provide key constraints on the physical conditions within relativistic jets, offering insights into particle acceleration mechanisms, high-energy radiation processes, and jet composition. The Rocket Experiment Demonstration of a Soft X-ray Polarimeter (REDSoX) will serve as a precursor to the Globe Orbiting Soft X-ray Polarimeter (GOSoX), a state-of-the-art polarimeter designed for observations in the soft X-ray range (0.2-0.4 keV), with a planned launch for 2030. REDSoX and GOSoX will be the first (and currently only) X-ray polarimeters capable of performing observations below 1 keV, opening a new observational window into high-energy polarimetry. In this work, we investigate the soft X-ray polarimetric properties of blazars expected in the REDSoX and GOSoX band and assess the detectability of different samples of sources: one drawn from the RoboPol Monitoring Program, a statistically complete sample of gamma-ray-selected blazars spanning the low-, intermediate-, and high-synchrotron-peaked sources, and one cross-matched between the CAZ (CRTS, ATLAS, and ZTF) and eROSITA catalogs. We estimated the expected source fluxes in the selected energy bands through spectral energy distribution fitting performed with the Bjet-MCMC tool. We investigated the expected polarimetric properties under three competing emission scenarios: leptonic, hadronic, and hybrid, each one predicting distinct observational signatures. We assessed the detectability of every source by comparing our polarization degree predictions with the instrument's minimum detectable polarization function (MDP). Finally, we computed the detection probability for each source. This work offers a set of promising blazar targets for future observations with GOSoX, highlighting the mission's potential for probing relativistic jet physics through soft X-ray polarimetry.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |

... (4 outputs left out)


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.26952-b31b1b.svg)](https://arxiv.org/abs/2611.26952) | **A High-Resolution Stellar Extinction Map of M31 from PHAT and PHAST Photometry**  |
|| M. Sun, et al. -- incl., <mark>Y. Wang</mark> |
|*Appeared on*| *2026-11-23*|
|*Comments*| *18 pages, 9 figures, accepted by The Astrophysical Journal Supplement Series (ApJS) on 19 November 2026*|
|**Abstract**|            High-resolution extinction maps of resolved external galaxies, such as M31, are crucial not only for accurately correcting dust extinction and reddening toward individual sources within the galaxy, but also for characterizing the distribution and evolution of interstellar dust in external galactic environments. We aim to construct a high-resolution and large-coverage stellar-traced extinction map of M31, based on multi-band photometric data from the HST PHAT and PHAST surveys. Using a stellar template library built from the PARSEC stellar evolution models and adopting the average extinction law of M31, we apply a matching method analogous to the SPar algorithm to derive A_V values for approximately 65 million stars. The extinction map is then generated using a HEALPix adaptive-resolution scheme. The resulting map covers approximately two-thirds of the star-forming disk of M31, with a maximum spatial resolution of 1.61 arcsec. The median extinction across the map is A_V = 1.24 mag. Comparison with previously published stellar- and dust-emission-based extinction maps shows broadly similar large-scale morphology, while differences remain in the extinction values. The extinction map further reveals fine-scale features such as inter-arm filaments, supernova-driven cavities, and spur-like structures. The map has been publicly released through the NADC platform and is expected to serve as an important observational basis for extinction correction in subsequent astronomical observations and for related M31 scientific studies.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.27031-b31b1b.svg)](https://arxiv.org/abs/2611.27031) | **Precise Radii of Visual Binaries with Dynamical Masses: $ξ$ Bootis $\&$ 70 Ophiuchi**  |
|| B. E. Flores, et al. -- incl., <mark>S. Kraus</mark> |
|*Appeared on*| *2026-11-23*|
|*Comments*| *26 pages, 13 figures, accepted for publication in The Astrophysical Journal*|
|**Abstract**|            A new orbital solution for $\xi$ Bootis (GJ 566 AB, P=152 yrs) and the recently revised orbital solution for 70 Ophiuchi (GJ 702 AB, P=88 yrs) yield component masses precise to 0.4\%. Their relatively wide separations ($>0.4^{\prime\prime}$) allow the components to be studied independently, making them ideal for testing the predictions of stellar evolutionary models. We use the six-telescope near-infrared MIRC-X and MYSTIC beam combiners at the CHARA Array to measure the angular diameters of all 4 stars. Combined with precise Gaia DR3 parallax values and bolometric fluxes, these interferometric measurements translate to stellar radii that are precise to $<2.4\%$ and effective temperatures precise to $<1.7\%$. We compare the measured radii and temperatures to four stellar evolutionary models (Lyon, Dartmouth, PARSEC, MIST). Over the mass range $0.67 - 0.91$ M$_\odot$, all models reproduce the observed properties within a few standard deviations, with no systematic offsets. The pairs are coeval and consistent with independent age estimates, further validating model predictions. With physical separations greater than 20 au, these pairs show no evidence of radius inflation, despite being spectroscopically classified as chromospherically active, and in the case of $\xi$ Bootis, probable members of the adolescent age ($\sim 0.4$ Gyr) Ursa Majoris Moving Group. In contrast, a set of 21 short-period ($P\lesssim$ 10 days) eclipsing binary stars with similar dynamical masses and metallicities are larger and often cooler than these same model predictions. Within this mass range, radius inflation appears to be associated with short-period binary star systems.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.27118-b31b1b.svg)](https://arxiv.org/abs/2611.27118) | **The relation between DCN/DNC ratios and gas temperature in late-stage massive star-forming regions using Yebes observations**  |
|| Y. Li, et al. -- incl., <mark>J. Li</mark> |
|*Appeared on*| *2026-11-23*|
|*Comments*| *Accepted for publication in A&A, 28 pages, 13 figures*|
|**Abstract**|            Deuterated molecules probe the physical and chemical evolution of molecular clouds and star-forming regions. Their abundances depend on temperature, density, and evolutionary stage. DCN and DNC are relatively abundant deuterated isomers that probe temperature-dependent deuterium chemistry. Although [HCN]/[HNC] is often thought to increase in warmer gas, the temperature dependence of [DCN]/[DNC] remains less well constrained. To investigate how [DCN]/[DNC] depends on the gas kinetic temperature, we observed the $J$=1-0 transitions of DCN, DNC, H$^{13}$CN, HN$^{13}$C, HC$^{15}$N, and H$^{15}$NC toward 101 late-stage massive star-forming regions using the Yebes 40m telescope. Gas kinetic temperatures were estimated from CH$_3$CCH 5-4, whose rotation temperature is closely related to the kinetic temperature of dense gas. These temperatures were adopted in the column density calculations and in the analysis of temperature-dependent abundance ratios. We find a moderate positive correlation between the abundance ratio [DCN]/[DNC] and kinetic temperature, which is statistically significant and indicates that DCN becomes progressively more abundant relative to DNC in warmer gas. A weak negative correlation with temperature is observed for $D_{\rm frac}({\rm HNC})$, whereas no statistically significant correlation is found for $D_{\rm frac}({\rm HCN})$. The [HCN]/[HNC] ratios show substantial scatter and no statistically significant correlation with temperature, possibly due in part to optical-depth effects in the $^{13}$C isotopologues and the limited sensitivity of the $^{15}$N isotopologue observations. These results suggest that [DCN]/[DNC] may be governed not only by HCN/HNC isomeric chemistry but also by temperature-dependent deuterium fractionation pathways and the chemical evolution of dense gas.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Heidelberg' keyword not found.</p> |


|||
|---:|:---|
| [![arXiv](https://img.shields.io/badge/arXiv-2611.27551-b31b1b.svg)](https://arxiv.org/abs/2611.27551) | **WISDOM Project -- XXIX. Catalogue of CO observations of nearby galaxies and the CO Tully-Fisher relation**  |
|| P. Dominiak, et al. -- incl., <mark>H. Zhang</mark> |
|*Appeared on*| *2026-11-23*|
|*Comments*| *60 pages, 5 figures, accepted by MNRAS*|
|**Abstract**|            We present Tully-Fisher relations (TFRs) that use carbon monoxide (CO) as the kinematic tracer and Wide-Field Infrared Survey Explorer (WISE) $W1$-band and Two Micron All Sky Survey (2MASS) $K_\text{s}$-band magnitudes as tracers of stellar luminosity. CO data were obtained using the Atacama Large Millimeter/submillimeter Array for the mm-Wave Interferometric Survey of Dark Object Masses (WISDOM) project for a sample of $102$ galaxies located at redshifts $z \lesssim 0.03$, with $81$ detections. The best-fitting reverse $W1$-band and $K_\text{s}$-band CO TFRs are $M_{W1}=(-7.18\pm0.93)\left[\log\left(\frac{W_{50}/\sin i}{\text{km~s}^{-1}}\right)-2.6\right]+(-24.04\pm0.13)$ and $M_{K_\text{s}}=(-5.95\pm0.62)\left[\log\left(\frac{W_{50}/\sin i}{\text{km~s}^{-1}}\right)-2.6\right]+(-24.01\pm0.10)$, respectively, where $W_{50}$ is the velocity width of each integrated CO spectrum at $50\%$ of its maximum, $i$ is the galaxy inclination and $M$ is the galaxy absolute magnitude. Our results are in excellent agreement with those of Tiley et al. (2016b), and together they constitute a robust local benchmark for the CO TFR. We however measure a shallower slope and a smaller zero-point offset compared to the relation of Tully & Pierce (2000). Due to the morphological diversity of our sample, we construct separate late-type and early-type galaxy (ETG) CO TFRs, finding that ETGs are $1.40\pm0.39$~mag fainter at $W1$ band and $0.89\pm0.28$~mag fainter at $K_\text{s}$ band at a given velocity width. We also generate and include a comprehensive catalogue of WISDOM CO data products for use in future molecular gas studies.         |
|<p style="color:green"> **ERROR** </p>| <p style="color:green">affiliation error: mpia.affiliation_verifications: 'Planck' keyword not found.</p> |

## Export documents

We now write the .md files and export relevant images

In [7]:
def export_markdown_summary(md: str, md_fname:str, directory: str):
    """Export MD document and associated relevant images"""
    import os
    import shutil
    import re

    if (os.path.exists(directory) and not os.path.isdir(directory)):
        raise RuntimeError(f"a non-directory file exists with name {directory:s}")

    if (not os.path.exists(directory)):
        print(f"creating directory {directory:s}")
        os.mkdir(directory)

    fig_fnames = (re.compile(r'\[Fig.*\]\((.*)\)').findall(md) + 
                  re.compile(r'\<img src="([^>\s]*)"[^>]*/>').findall(md))
    print("found figures", fig_fnames)
    for fname in fig_fnames:
        if 'http' in fname:
            # No need to copy online figures
            continue
        if not os.path.exists(fname):
            print("file not found", fname)
            continue
        print("copying ", fname, "to", directory)
        destdir = os.path.join(directory, os.path.dirname(fname))
        destfname = os.path.join(destdir, os.path.basename(fname))
        try:
            os.makedirs(destdir)
        except FileExistsError:
            pass
        shutil.copy(fname, destfname)
    with open(os.path.join(directory, md_fname), 'w') as fout:
        fout.write(md)
    print("exported in ", os.path.join(directory, md_fname))
    [print("    + " + os.path.join(directory,fk)) for fk in fig_fnames]

In [8]:
for paper_id, md in documents:
    export_markdown_summary(md, f"{paper_id:s}.md", '_build/html/')

found figures ['tmp_2611.27082/./Figure_detections_MSNoTi.png', 'tmp_2611.27082/./Figure_TPchachan_MSNoTi.png', 'tmp_2611.27082/./Figure_CtoO_MSNoTi.png']
copying  tmp_2611.27082/./Figure_detections_MSNoTi.png to _build/html/
copying  tmp_2611.27082/./Figure_TPchachan_MSNoTi.png to _build/html/
copying  tmp_2611.27082/./Figure_CtoO_MSNoTi.png to _build/html/
exported in  _build/html/2611.27082.md
    + _build/html/tmp_2611.27082/./Figure_detections_MSNoTi.png
    + _build/html/tmp_2611.27082/./Figure_TPchachan_MSNoTi.png
    + _build/html/tmp_2611.27082/./Figure_CtoO_MSNoTi.png
found figures ['tmp_2611.27643/./DRTau_plottadisk_all_v2_mod.png', 'tmp_2611.27643/./figure1.png', 'tmp_2611.27643/./polarimetric_data_annotated_norot_2_mod.png']
copying  tmp_2611.27643/./DRTau_plottadisk_all_v2_mod.png to _build/html/
copying  tmp_2611.27643/./figure1.png to _build/html/
copying  tmp_2611.27643/./polarimetric_data_annotated_norot_2_mod.png to _build/html/
exported in  _build/html/2611.27643.md

## Display the papers

Not necessary but allows for a quick check.

In [9]:
[display(Markdown(k[1])) for k in documents];

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\vdag}{(v)^\dagger}$
$\newcommand\aastex{AAS\TeX}$
$\newcommand\latex{La\TeX}$
$\newcommand\thefigure{A\arabic{figure}}$</div>



<div id="title">

# IGRINS-2 Reveals an Oxygen-rich Dayside Atmosphere of WASP-12 b: Resolving the C/O Controversy

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.27082-b31b1b.svg)](https://arxiv.org/abs/2611.27082)<mark>Appeared on: 2026-11-23</mark> -  _12 pages, 8 figures, 1 table. Accepted for publication in ApJL_

</div>
<div id="authors">

C. Kye, et al. -- incl., <mark>P. Smith</mark>

</div>
<div id="abstract">

**Abstract:** Recent studies have shown that measurements of hot-Jupiter atmospheric composition, including carbon-to-oxygen (C/O) and refractory-to-volatile (R/V) abundance ratios, provide powerful diagnostics of the chemistry and formation of gas giant planets.Important questions remain about the atmospheric composition of the ultra-hot Jupiter WASP-12 b: its dayside C/O ratio remains under debate, and its R/V ratio is not yet well constrained.We present high-resolution (R $\sim$ 45,000) near-infrared dayside emission spectroscopy of the ultra-hot Jupiter WASP-12 b with IGRINS-2 on Gemini North during post-eclipse phases.Cross-correlation analysis reveals volatile molecular signals of $H_2$ O and CO together with tentative signals of OH and gas-phase refractories in the planetary atmosphere.The planetary signals appear in emission, indicating a thermal inversion in the dayside atmosphere.A chemically consistent Bayesian retrieval yields C/O $=0.72^{+0.08}_{-0.10}$ , disfavoring carbon-rich dayside solutions with C/O $>1$ at greater than $5\sigma$ .This independent, line-resolved measurement supports an oxygen-rich atmosphere, providing a new constraint on the long-standing C/O controversy surrounding WASP-12 b.Our retrieval also yields a host-star-normalized R/V ratio, $[\mathrm{R/V}]_{*}=0.17^{+0.22}_{-0.23}$ , which is consistent with those of other ultra-hot Jupiters similarly characterized with IGRINS/IGRINS-2.Although the sample remains small, this consistency suggests a shared atmospheric chemistry among these planets, potentially reflecting similar proportions of accreted volatile and refractory material.

</div>

<div id="div_fig1">

<img src="tmp_2611.27082/./Figure_detections_MSNoTi.png" alt="Fig4" width="100%"/>

**Figure 4. -** **(a)**: Temperature--pressure profile (dashed black curve) and volume-mixing-ratio profiles of the selected opacity sources. **(b)**: Corresponding model planet-to-star flux ratio spectra over the IGRINS-2 wavelength range, showing the all-species spectrum, the individual volatile contributions from CO, H$_{2}$O, and OH, and the combined contribution from the refractory species. Each spectrum has a vertical offset for clarity. **(c)**: Cross-correlation trail for the all-species template as a function of orbital phase and radial velocity. The red dashed curve indicates the expected planetary radial velocity track. **(d--h)**: Integrated cross-correlation S/N maps in the $(v_{\mathrm{sys}}, K_{p})$ plane for the all-species template (**d**), the individual CO, H$_{2}$O, and OH templates (**e, f, g**, respectively), and the combined refractory template (**h**). White dashed lines indicate the expected planetary $v_{\mathrm{sys}}$ and $K_p$ values, and black crosses mark the position of the maximum S/N in each integrated map. (*fig:detection*)

</div>
<div id="div_fig2">

<img src="tmp_2611.27082/./Figure_TPchachan_MSNoTi.png" alt="Fig5" width="100%"/>

**Figure 5. -** **(a)**: Retrieved dayside temperature--pressure profile of WASP-12 b from the chemically consistent retrieval. The solid blue curve shows the median posterior profile, and the blue shaded region represents the $1\sigma$ credible range at each pressure level. The gray shading illustrates the wavelength-dependent pressure range probed by the emission. **(b)**: Comparison of the retrieved abundance-ratio posterior for WASP-12 b with the formation scenario predictions of [Chachan, et. al (2023)](https://ui.adsabs.harvard.edu/abs/2023ApJ...943..112C) using [C/R]$_{*}$ versus [R/H]$_{*}$. **(c)**: [O/R]$_{*}$ versus [R/H]$_{*}$. Light-blue points show posterior samples, black contours indicate the 1, 2, and 3$\sigma$ regions of the posterior distribution, and the colored tracks correspond to the eight formation scenarios from [Chachan, et. al (2023)](https://ui.adsabs.harvard.edu/abs/2023ApJ...943..112C). Here, R denotes refractory elements, and all abundance ratios are shown relative to the host star composition. (*fig:TPchachan*)

</div>
<div id="div_fig3">

<img src="tmp_2611.27082/./Figure_CtoO_MSNoTi.png" alt="Fig1" width="100%"/>

**Figure 1. -** Posterior distribution of the retrieved C/O ratio of WASP-12 b. The solid black, red dashed, and orange dashed lines respectively indicate the median, $1\sigma$ credible range, and $5\sigma$ upper credible limit of C/O=0.97. The black star marks the host star C/O ratio derived from the stellar abundances reported by [Fossati, et. al (2010)](https://ui.adsabs.harvard.edu/abs/2010ApJ...720..872F), with the $1\sigma$ error bar. The red arrow indicates the lower limit of C/O ratios from [Madhusudhan, et. al (2011)](https://ui.adsabs.harvard.edu/abs/2011Natur.469...64M), while the green and blue bars show the $1\sigma$ ranges from [Line, et. al (2014)](https://ui.adsabs.harvard.edu/abs/2014ApJ...783...70L) and [Kreidberg, et. al (2015)](https://ui.adsabs.harvard.edu/abs/2015ApJ...814...66K). (*fig:ctoo_post*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.27082"></div>

<div class="macros" style="visibility:hidden;">
$\newcommand{\ensuremath}{}$
$\newcommand{\xspace}{}$
$\newcommand{\object}[1]{\texttt{#1}}$
$\newcommand{\farcs}{{.}''}$
$\newcommand{\farcm}{{.}'}$
$\newcommand{\arcsec}{''}$
$\newcommand{\arcmin}{'}$
$\newcommand{\ion}[2]{#1#2}$
$\newcommand{\textsc}[1]{\textrm{#1}}$
$\newcommand{\hl}[1]{\textrm{#1}}$
$\newcommand{\footnote}[1]{}$
$\newcommand{\MJup}{M_{\mathrm{Jup}}\xspace}$
$\newcommand{\RJup}{R_{\mathrm{Jup}}\xspace}$
$\newcommand{\RSun}{R_{\odot}\xspace}$
$\newcommand{\MSun}{M_{\odot}\xspace}$
$\newcommand{\LSun}{L_{\odot}\xspace}$
$\newcommand{\teff}{T_{e\!f\!f}\xspace}$
$\newcommand{\logg}{log~\emph{g}\xspace}$
$\newcommand{\mic}{\mum\xspace}$
$\newcommand{\as}{\hbox{^{\prime\prime}}\xspace}$
$\newcommand{\ha}{H\alpha}$
$\newcommand{\sv}{SHARK-VIS}$
$\newcommand{\labelitemi}{\bullet}$
$\newcommand\tablebib{#1}$</div>



<div id="title">

# Hints of planet formation in progress around DR Tau from observations with LBT SHARK-NIR and SHARK-VIS

</div>
<div id="comments">

[![arXiv](https://img.shields.io/badge/arXiv-2611.27643-b31b1b.svg)](https://arxiv.org/abs/2611.27643)<mark>Appeared on: 2026-11-23</mark> -  _12 pages; 10 Figures; Accepted for publications on A&A_

</div>
<div id="authors">

D. Mesa, et al. -- incl., <mark>L. Mohr</mark>, <mark>M. V. Pallauta</mark>

</div>
<div id="abstract">

**Abstract:** Protoplanetary disks are the primary sites of planet formation. However, the direct imaging of protoplanets still embedded within their parental disks remains a significant challenge, with only a small number of confirmed detections to date and several candidates awaiting verification. We search for embedded planets in the protoplanetary disk of the young star DR Tau by targeting accretion signatures from forming sub-stellar objects. Using narrow-band imaging, we observe both visible $H\alpha$ and near-infrared Paschen- $\beta$ ( $Pa\beta$ ) emission to identify actively accreting companions. To this end, we performed high-contrast observations of DR Tau using the SHARK-VIS (0.40–0.95 $\mic$ ) and SHARK-NIR (0.96–1.70 $\mic$ ) imagers at the LBT. The study uses a combination of narrow-band filters to isolate accretion signatures and broad-band filters for multi-wavelength characterization in both the visible and near-infrared. Our observations recover the primary disk structures previously identified in the literature, albeit at low signal-to-noise (S/N). We identify two compact features with S/N between 6 and 13 in both narrow- and broad-band filters. The first, an elongated structure, matches previous SPHERE detections, while the second, more compact feature, is now confirmed despite its low S/N in earlier datasets. A comparison over a seven-year baseline confirms the stability of both structures. Their observed orbital motion is consistent with Keplerian rotation, potentially suggesting the presence of an undetected planetary-mass companion ( $\sim$ few $M_{\text{Jup}}$ ) at the location of the second feature. However, our analysis does not confirm H $\alpha$ or Pa $\beta$ emission originating from localized accretion. Instead, we conclude that the observed flux is likely stellar light reflected by dust at these locations and discuss alternative mechanisms that could account for such detections.

</div>

<div id="div_fig1">

<img src="tmp_2611.27643/./DRTau_plottadisk_all_v2_mod.png" alt="Fig7" width="100%"/>

**Figure 7. -** Comparison of the SPHERE and SHARK results for the disk spirals around DR Tau. _ Left panel:_ Final image for the SPHERE polarized data from [Mesa, et. al (2022)](https://ui.adsabs.harvard.edu/abs/2022A&A...658A..63M). Following that paper we  superimposed red dashed lines to highlight the main disk structures and we have tagged them with SS standing for southern spiral, NES standing for north-eastern spiral, and ALS standing for arc-like structure. _ Right panel: _ Final image from the SHARK-NIR broad-band H data with superimposed dashed lines on the same structures of the SPHERE case to highlight that the same disk structures have been recovered. The dark dashed line on the left represents
a possible arm-like structure that was not visible in the SPHERE data. (*f:diskall*)

</div>
<div id="div_fig2">

<img src="tmp_2611.27643/./figure1.png" alt="Fig6" width="100%"/>

**Figure 6. -** Final images obtained both for SHARK-NIR (top panels) and SHARK-VIS (bottom panels). In the top left panel we display
the final image obtained using the the narrow-band _ Pa$\beta$_ filter while in the top right panel we display the final
image obtained using the broad-band $H$ filter. Both the SHARK-NIR images have been obtained by applying a PCA-ADI method,
subtracting five principal components. The white crosses represent the positions of the star in both images. In the bottom
left panel we display the final image obtained using the _ H$\alpha$_ narrow-band filter of SHARK-VIS while in the bottom
right panel we display the final image obtained using the "$continuum$" filter of SHARK-VIS. The same structures visible in
the SHARK-NIR images are also visible in these images. The blue crosses represents the position of the star. (*f:DRTau_finimg*)

</div>
<div id="div_fig3">

<img src="tmp_2611.27643/./polarimetric_data_annotated_norot_2_mod.png" alt="Fig9" width="100%"/>

**Figure 9. -** _ Left panel_: Pseudo-ADI polarimetric $Q_\phi$ image of DR Tau obtained from the SPHERE H broad band polarimetric data acquired in 2018. The position of S2 (at $\sim$74 au from the star) is marked with a cyan circle whose size approximately reproduces where the area with micron-sized dust is observed and a circular orbit with a solid green circle. The inner edge of the cavity observed in the ALMA continuum image is shown as a dashed green circle.  We also show vectors from the star rotated by $\pm$60 degrees with respect to S2; the intersections of these vectors with the orbit of S2 identify the L4 (leading) and L5 (trailing) Lagrangian points, where we might expect evidence of material co-orbiting S2. This might be expected in correspondence of the main phase of accretion on a planet located in S2. Finally, we signed the primary (NES) and the secondary spiral (SS) arm related to the disk perturbation if due to a planet at the location of S2. _ Right panel:_ the ALMA continuum image from band 6 data acquired within program 2022.1.01365.S (PI Jane Huang) on the same scale as for the SPHERE image. The expected position of S2 (cyan circle) takes into account the possible orbital motion around the star. (*f:annotated*)

</div><div id="qrcode"><img src=https://api.qrserver.com/v1/create-qr-code/?size=100x100&data="https://arxiv.org/abs/2611.27643"></div>

# Create HTML index

In [10]:
from datetime import datetime, timedelta, timezone
from glob import glob
import os

files = glob('_build/html/*.md')
days = 7
now = datetime.today()
res = []
for fk in files:
    stat_result = os.stat(fk).st_ctime
    modified = datetime.fromtimestamp(stat_result, tz=timezone.utc).replace(tzinfo=None)
    delta = now.today() - modified
    if delta <= timedelta(days=days):
        res.append((delta.seconds, fk))
res = [k[1] for k in reversed(sorted(res, key=lambda x:x[1]))]
npub = len(res)
print(len(res), f" publications files modified in the last {days:d} days.")
# [ print('\t', k) for k in res ];

159  publications files modified in the last 7 days.


In [11]:
import datetime
from glob import glob

def get_last_n_days(lst, days=1):
    """ Get the documents from the last n days """
    sorted_lst = sorted(lst, key=lambda x: x[1], reverse=True)
    for fname, date in sorted_lst:
        if date >= str(datetime.date.today() - datetime.timedelta(days=days)):
            yield fname

def extract_appearance_dates(lst_file):
    dates = []

    def get_date(line):
        return line\
            .split('Appeared on:')[-1]\
            .split('</mark>')[0].strip()

    for fname in lst:
        with open(fname, 'r') as f:
            found_date = False
            for line in f:
                if not found_date:
                    if "Appeared on" in line:
                        found_date = True
                        dates.append((fname, get_date(line)))
                else:
                    break
    return dates

from glob import glob
lst = glob('_build/html/*md')
days = 7
dates = extract_appearance_dates(lst)
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last {days:d} days.")

18  publications in the last 7 days.


In [12]:
def create_carousel(npub=4):
    """ Generate the HTML code for a carousel with `npub` slides """
    carousel = ["""  <div class="carousel" """,
                """       data-flickity='{ "autoPlay": 10000, "adaptiveHeight": true, "resize": true, "wrapAround": true, "pauseAutoPlayOnHover": true, "groupCells": 1 }' id="asyncTypeset">"""
                ]
    
    item_str = """    <div class="carousel-cell"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        carousel.append(item_str.format(k=k))
    carousel.append("  </div>")
    return '\n'.join(carousel)

def create_grid(npub=4):
    """ Generate the HTML code for a flat grid with `npub` slides """
    grid = ["""  <div class="grid"> """,
                ]
    
    item_str = """    <div class="grid-item"> <div id="slide{k}" class="md_view">Content {k}</div> </div>"""
    for k in range(1, npub + 1):
        grid.append(item_str.format(k=k))
    grid.append("  </div>")
    return '\n'.join(grid)

In [13]:
carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "7-day archives" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
with open("_build/html/index_7days.html", 'w') as fout:
    fout.write(page)

In [14]:
# redo for today
days = 1
res = list(get_last_n_days(dates, days))
npub = len(res)
print(len(res), f" publications in the last day.")

carousel = create_carousel(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("daily_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- carousel:s --%}", carousel)\
               .replace("{%-- suptitle:s --%}",  "Daily" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(carousel, docs, slides)
# print(page)
with open("_build/html/index_daily.html", 'w') as fout:
    fout.write(page)

2  publications in the last day.


In [15]:
# Create the flat grid of the last N papers (fixed number regardless of dates)
from itertools import islice 

npub = 6
res = [k[0] for k in (islice(reversed(sorted(dates, key=lambda x: x[1])), 6))]
print(len(res), f" {npub} publications selected.")

grid = create_grid(npub)
docs = ', '.join(['"{0:s}"'.format(k.split('/')[-1]) for k in res])
slides = ', '.join([f'"slide{k}"' for k in range(1, npub + 1)])

with open("grid_template.html", "r") as tpl:
    page = tpl.read()
    page = page.replace("{%-- grid-content:s --%}", grid)\
               .replace("{%-- suptitle:s --%}",  f"Last {npub:,d} papers" )\
               .replace("{%-- docs:s --%}", docs)\
               .replace("{%-- slides:s --%}", slides)
    
# print(grid, docs, slides)
# print(page)
with open("_build/html/index_npub_grid.html", 'w') as fout:
    fout.write(page)

6  6 publications selected.
